# Step 2: Filter notebook
Ce notebook filtre les attributs et uniforme le format afin qu'ils soient prêt à être intégré à l'indice. 
Les filtres sont évalués au cas par cas selon la pertinence.

In [1]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
import rasterio
import numpy as np
import os
import folium
import json
from folium import Choropleth, CircleMarker, GeoJson
import branca.colormap as cm
from IPython.display import display
pd.set_option('display.max_columns', None)

from feature_to_network import *

## Imports

#### Import des segments

In [2]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters : GG or GE, bike or walk")
territory = 'GG' # GG or GE
network = "bike"  # bike ONLY in the notebook, go to 2_1_Filter_features_walk.ipynb for walk

input_file_path = '../../Data/input'
output_step1_path=f"../../Data/output/{network}/{territory}/step-1"
output_step2_path=f"../../Data/output/{network}/{territory}/step-2"
output_step3_path=f"../../Data/output/{network}/{territory}/step-3"
save_path = f"../../Data/output/{network}/{territory}/step-2"

save_filtered_attributes = True
attribute_source_path = f"source_path_{territory}"
file_name = f"file_name_{territory}"  # column name in attributs_info excel

# Load segments GeoDataFrame (with 'segment_id')
print("Loading segments...")
# reLoad bike segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_all_segments.parquet"))
segmented_net["geometry"] = segmented_net["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
segmented_net = segmented_net.to_crs(operation_crs)


Set parameters : GG or GE, bike or walk
Loading segments...


In [3]:
import geopandas as gpd
from shapely.geometry.base import BaseGeometry

# Clean geometries safely
def clean_any_geom(geom):
    if geom is None:
        return None
    if not isinstance(geom, BaseGeometry):
        return None
    if geom.is_valid:
        return geom
    try:
        fixed = geom.buffer(0)
        if fixed is None or fixed.is_empty:
            return geom
        return fixed
    except Exception:
        return geom
        




# Define a function to save the filtered data
def save(save_filtered_attributes, row, gdf, attribute):

    if not save_filtered_attributes:
        print("Note : Save option is disabled.")
        return

    # Parse save formats: comma-separated list allowed
    raw = str(row.get('save_format', '') or '')
    formats = [f.strip().lower() for f in raw.split(',') if f.strip()]
    if not formats:
        formats = ['csv']  # default fallback

    for fmt in formats:
        try:
            if fmt == 'parquet':
                dirpath = f'{output_step2_path}/parquet_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_parquet(f"{dirpath}/{attribute}.parquet")
                print(f"Filtered data saved for attribute: {attribute} in format: parquet")

            elif fmt == 'gpkg' or fmt == 'geopackage':
                dirpath = f'{output_step2_path}/gpkg_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_file(f"{dirpath}/{attribute}.gpkg", driver="GPKG")
                print(f"Filtered data saved for attribute: {attribute} in format: gpkg")

            elif fmt == 'csv':
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                # to_csv may not preserve geometry consistently; keep original behavior
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Filtered data saved for attribute: {attribute} in format: csv")

            else:
                # Unknown format -> fallback to csv and warn
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Warning: Unknown save format '{fmt}' for attribute {attribute}. Data saved as csv.")
        except Exception as e:
            print(f"Error saving {attribute} as {fmt}: {e}")


#### Import des attributs 

In [4]:
# Classeur des attributs vélo (mettre à jour le nom si le fichier WIP est renommé)
ATTRIBUTS_INFO_FILE = "attributs_info_bike_WIP.xlsx"
attributs_info = pd.read_excel(f"{input_file_path}/attributs/{ATTRIBUTS_INFO_FILE}", sheet_name="attributs_info")

# include_in_index_<territoire> (ancien format : include_in_index). Excel peut lire
# ces colonnes en booléens ou en texte ('true'/'false') : on normalise.
include = f"include_in_index_{territory}" if f"include_in_index_{territory}" in attributs_info.columns else "include_in_index"
_norm = lambda col: col.astype(str).str.strip().str.lower()
compute_name = f"compute_step2_{territory}"
# Ligne gardée si elle est dans l'indice OU marquée compute_step2_<T> = True
# (comptages intermédiaires hors indice : n_traversees, n_traversees_feux), comme en 2_2.
_keep = _norm(attributs_info[include]).ne("false")
if compute_name in attributs_info.columns:
    _keep |= _norm(attributs_info[compute_name]).eq("true")
attributs_info = attributs_info[_keep & attributs_info["attribute"].notna()]

attributs_info

,Class,meta_attribute,attribute,include_in_index_GG,include_in_index_GE,source_type_GE,source_path_GE,file_name_GE,source_type_VD,source_path_VD,file_name_VD,source_type_FR,source_path_FR,file_name_FR,source_type_GG,source_path_GG,file_name_GG,merge_rule_GG,initial_weight,class_weight,buffer_size_GG,buffer_size_GE,impact_attribut,geometry_type_GE,how_GE,value_column_GE,geometry_type_GG,how_GG,value_column_GG,clip_GG,clip_GE,missing_value_strategy,filter_column,filter_values,crs,save_format,Unnamed: 36,commentaire,extend_m_GG,compute_step2_GG
0,attractivite,amenite,amenite,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,0.5,75,75.0,favorable,point,count,NaN,point,count,NaN,4.0,4.0,zero_if_missing,filtered,1.0,2056,"parquet, csv, gpkg",NaN,NaN,NaN,NaN
2,attractivite,connectivite,connectivite,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,0.5,10,10.0,favorable,line,sum,conn_branching_in_buffer,line,sum,conn_branching_in_buffer,7.0,7.0,zero_if_missing,filtered,1.0,2056,"parquet, csv, gpkg",NaN,NaN,NaN,NaN
3,attractivite,pente,pente,True,True,opendata,attributs/GE/topographie,pente_2056.tif,NaN,NaN,NaN,NaN,NaN,NaN,opendata,attributs/GG/topographie,pente_2056.tif,prefer_opendata,0.5,0.5,10,10.0,defavorable,line,sum,slope_pct_abs_smoothed,line,sum,slope_pct_abs_smoothed,75.0,75.0,keep_nan,filtered,1.0,2056,"parquet, csv, gpkg",NaN,top,NaN,NaN
4,confort,lac_cours_deau,eau,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,0.5,50,50.0,favorable,polygon,presence,NaN,polygon,presence,NaN,NaN,NaN,zero_if_missing,filtered,1.0,2056,"parquet, csv, gpkg",NaN,NaN,NaN,NaN
5,confort,temperature,temperature,True,True,opendata,attributs/GE/temperature,20250629_102250.LST.tif,NaN,NaN,NaN,NaN,NaN,NaN,opendata,attributs/GG/temperature,20250629_102250.LST.tif,prefer_opendata,0.0,0.0,250,250.0,defavorable,point,raster,temperature,point,raster,temperature,40.0,40.0,keep_nan,filtered,1.0,2056,"parquet, csv, gpkg",NaN,NaN,NaN,NaN
6,confort,air,air,True,True,opendata,attributs/GE/air,NO2_Extraction_2025.tif,NaN,NaN,NaN,NaN,NaN,NaN,opendata,attributs/GG/air,NO2_Extraction_2025.tif,prefer_opendata,0.0,0.0,750,750.0,defavorable,point,raster,no_2,point,raster,no_2,NaN,NaN,keep_nan,filtered,1.0,2056,"parquet, csv, gpkg",NaN,NaN,NaN,NaN
7,confort,landuse_env,alentours,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.0,0.0,50,50.0,favorable,polygon,area_ratio,NaN,polygon,area_ratio,NaN,NaN,NaN,zero_if_missing,filtered,1.0,2056,"parquet, csv, gpkg",NaN,NaN,NaN,NaN
8,confort,canopee,canopee,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,sitg,attributs/GG/canopee,SIPV_MN_CARTO_GG.gpkg,prefer_opendata,0.5,0.5,25,25.0,favorable,polygon,area_ratio,NaN,polygon,area_ratio,NaN,NaN,NaN,zero_if_missing,filtered,1.0,2056,"parquet, csv, gpkg",NaN,"GG : couverture du sol SITG, forêts et végétat...",NaN,NaN
9,confort,environnement_ouvert_naturel,environnement_ouvert_naturel,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sitg,attributs/GG/environnement_ouvert_naturel,SIPV_MN_CARTO_GG.gpkg,NaN,0.5,0.5,30,NaN,favorable,NaN,NaN,NaN,polygon,area_ratio,NaN,NaN,NaN,zero_if_missing,filtered,1.0,2056,"parquet, csv, gpkg",NaN,"GG : couverture du sol SITG, prairies/cultures...",NaN,NaN
10,equipement,stationnement_velo,stationnement_velo,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,0.5,30,30.0,favorable,point,sum,score,point,sum,score,5.0,5.0,zero_if_missing,filtered,1.0,2056,"parquet, csv, gpkg",NaN,NaN,NaN,NaN


In [5]:
attributs_info

,Class,meta_attribute,attribute,include_in_index_GG,include_in_index_GE,source_type_GE,source_path_GE,file_name_GE,source_type_VD,source_path_VD,file_name_VD,source_type_FR,source_path_FR,file_name_FR,source_type_GG,source_path_GG,file_name_GG,merge_rule_GG,initial_weight,class_weight,buffer_size_GG,buffer_size_GE,impact_attribut,geometry_type_GE,how_GE,value_column_GE,geometry_type_GG,how_GG,value_column_GG,clip_GG,clip_GE,missing_value_strategy,filter_column,filter_values,crs,save_format,Unnamed: 36,commentaire,extend_m_GG,compute_step2_GG
0,attractivite,amenite,amenite,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,0.5,75,75.0,favorable,point,count,NaN,point,count,NaN,4.0,4.0,zero_if_missing,filtered,1.0,2056,"parquet, csv, gpkg",NaN,NaN,NaN,NaN
2,attractivite,connectivite,connectivite,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,0.5,10,10.0,favorable,line,sum,conn_branching_in_buffer,line,sum,conn_branching_in_buffer,7.0,7.0,zero_if_missing,filtered,1.0,2056,"parquet, csv, gpkg",NaN,NaN,NaN,NaN
3,attractivite,pente,pente,True,True,opendata,attributs/GE/topographie,pente_2056.tif,NaN,NaN,NaN,NaN,NaN,NaN,opendata,attributs/GG/topographie,pente_2056.tif,prefer_opendata,0.5,0.5,10,10.0,defavorable,line,sum,slope_pct_abs_smoothed,line,sum,slope_pct_abs_smoothed,75.0,75.0,keep_nan,filtered,1.0,2056,"parquet, csv, gpkg",NaN,top,NaN,NaN
4,confort,lac_cours_deau,eau,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,0.5,50,50.0,favorable,polygon,presence,NaN,polygon,presence,NaN,NaN,NaN,zero_if_missing,filtered,1.0,2056,"parquet, csv, gpkg",NaN,NaN,NaN,NaN
5,confort,temperature,temperature,True,True,opendata,attributs/GE/temperature,20250629_102250.LST.tif,NaN,NaN,NaN,NaN,NaN,NaN,opendata,attributs/GG/temperature,20250629_102250.LST.tif,prefer_opendata,0.0,0.0,250,250.0,defavorable,point,raster,temperature,point,raster,temperature,40.0,40.0,keep_nan,filtered,1.0,2056,"parquet, csv, gpkg",NaN,NaN,NaN,NaN
6,confort,air,air,True,True,opendata,attributs/GE/air,NO2_Extraction_2025.tif,NaN,NaN,NaN,NaN,NaN,NaN,opendata,attributs/GG/air,NO2_Extraction_2025.tif,prefer_opendata,0.0,0.0,750,750.0,defavorable,point,raster,no_2,point,raster,no_2,NaN,NaN,keep_nan,filtered,1.0,2056,"parquet, csv, gpkg",NaN,NaN,NaN,NaN
7,confort,landuse_env,alentours,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.0,0.0,50,50.0,favorable,polygon,area_ratio,NaN,polygon,area_ratio,NaN,NaN,NaN,zero_if_missing,filtered,1.0,2056,"parquet, csv, gpkg",NaN,NaN,NaN,NaN
8,confort,canopee,canopee,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,sitg,attributs/GG/canopee,SIPV_MN_CARTO_GG.gpkg,prefer_opendata,0.5,0.5,25,25.0,favorable,polygon,area_ratio,NaN,polygon,area_ratio,NaN,NaN,NaN,zero_if_missing,filtered,1.0,2056,"parquet, csv, gpkg",NaN,"GG : couverture du sol SITG, forêts et végétat...",NaN,NaN
9,confort,environnement_ouvert_naturel,environnement_ouvert_naturel,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sitg,attributs/GG/environnement_ouvert_naturel,SIPV_MN_CARTO_GG.gpkg,NaN,0.5,0.5,30,NaN,favorable,NaN,NaN,NaN,polygon,area_ratio,NaN,NaN,NaN,zero_if_missing,filtered,1.0,2056,"parquet, csv, gpkg",NaN,"GG : couverture du sol SITG, prairies/cultures...",NaN,NaN
10,equipement,stationnement_velo,stationnement_velo,True,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,0.5,30,30.0,favorable,point,sum,score,point,sum,score,5.0,5.0,zero_if_missing,filtered,1.0,2056,"parquet, csv, gpkg",NaN,NaN,NaN,NaN


**Attributs OSM**

**Attribut stationnement velo**


Amélioration: donner un meilleur score selon la capacité, présence abris ou non

In [6]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'stationnement_velo'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1

# --- Capacity cleaning (handles empty strings, None, non-numeric) ---
gdf["capacity_num"] = pd.to_numeric(
    gdf["capacity"].astype(str).str.strip().replace({"": None, "nan": None, "None": None}),
    errors="coerce"
)

# --- Score using capacity, fallback to 1 ---
gdf["score"] = gdf["capacity_num"].fillna(1)

# (optional) avoid zero/negative capacities becoming weird scores
gdf.loc[gdf["score"] <= 0, "score"] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: stationnement_velo
Filters applied
Proportion of features stationnement_velo kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: stationnement_velo in format: parquet
Filtered data saved for attribute: stationnement_velo in format: csv
Filtered data saved for attribute: stationnement_velo in format: gpkg


**Attribut location**

In [7]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'location'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: location
Filters applied
Proportion of features location kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: location in format: parquet
Filtered data saved for attribute: location in format: csv
Filtered data saved for attribute: location in format: gpkg


**Attribut zone apaisée**

In [8]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'zone_apaisee'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[(gdf.speed_kph > 5) & (gdf.speed_kph < 30), 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: zone_apaisee
Filters applied
Proportion of features zone_apaisee kept after filtering:
filtered
0    0.918803
1    0.081197
Name: proportion, dtype: float64
Filtered data saved for attribute: zone_apaisee in format: parquet
Filtered data saved for attribute: zone_apaisee in format: csv
Filtered data saved for attribute: zone_apaisee in format: gpkg


**Atttribut Vitesse motorisée**

In [9]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'vitesse_motorisee'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.speed_kph>25, 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: vitesse_motorisee
Filters applied
Proportion of features vitesse_motorisee kept after filtering:
filtered
1    0.803054
0    0.196946
Name: proportion, dtype: float64
Filtered data saved for attribute: vitesse_motorisee in format: parquet
Filtered data saved for attribute: vitesse_motorisee in format: csv
Filtered data saved for attribute: vitesse_motorisee in format: gpkg


**Attribut Contre-sens cyclable**

In [10]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'sens_inverse'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.TYPE_AMEN=="Contresens", 'filtered'] = 1
gdf.loc[gdf.TYPE_AMEN=="Dérogation 2R", 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: sens_inverse
Filters applied
Proportion of features sens_inverse kept after filtering:
filtered
0    0.79114
1    0.20886
Name: proportion, dtype: float64
Filtered data saved for attribute: sens_inverse in format: parquet
Filtered data saved for attribute: sens_inverse in format: csv
Filtered data saved for attribute: sens_inverse in format: gpkg


**Attribut Giratoire**

In [11]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'giratoire'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1


###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: giratoire
Filters applied
Proportion of features giratoire kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: giratoire in format: parquet
Filtered data saved for attribute: giratoire in format: csv
Filtered data saved for attribute: giratoire in format: gpkg


**Attribut revêtement**

In [12]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'revetement'


###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
surface_score = {
    "asphalt": 1,
    "concrete": 1,
    "concrete:plates": 1,
    "concrete:lanes": 1,
    "paved": 0,
    "paving_stones": 0,
    "sett": 0,
    "metal": 0,
    "wood": 0,

    "fine_gravel": 0,
    "compacted": 0,
    "gravel": 0,
    "pebblestone": 0,

    "ground": 0,
    "dirt": 0,
    "earth": 0,
    "grass": 0,
    "grass_paver": 0,
    "sand": 0,
    "rock": 0,
    "unpaved": 0,
    "woodchips": 0,
}


gdf["surface_score"] = gdf["surface"].map(surface_score)
gdf.loc[gdf["surface_score"] == 1, "filtered"] = 1


###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: revetement
Filters applied
Proportion of features revetement kept after filtering:
filtered
1    0.811192
0    0.188808
Name: proportion, dtype: float64
Filtered data saved for attribute: revetement in format: parquet
Filtered data saved for attribute: revetement in format: csv
Filtered data saved for attribute: revetement in format: gpkg


**Attribut Aménité**

In [13]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'amenite'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")
# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
rez_actif_set = {
"restaurant","cafe","bar","pub","fast_food","ice_cream","biergarten",
"bank","atm","bureau_de_change",
"cinema","theatre","nightclub","casino",
"marketplace","vending_machine"
}

gdf.loc[gdf["amenity"].isin(rez_actif_set), "filtered"] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)


Data initialized
Processing attribute: amenite
Filters applied
Proportion of features amenite kept after filtering:
filtered
0    0.861053
1    0.138947
Name: proportion, dtype: float64
Filtered data saved for attribute: amenite in format: parquet
Filtered data saved for attribute: amenite in format: csv
Filtered data saved for attribute: amenite in format: gpkg


**Attribut borne réparation**

In [14]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'borne_reparation'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: borne_reparation
Filters applied
Proportion of features borne_reparation kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: borne_reparation in format: parquet
Filtered data saved for attribute: borne_reparation in format: csv
Filtered data saved for attribute: borne_reparation in format: gpkg


**Attribut éclairage**

In [15]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'eclairage'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.lit=='yes', 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: eclairage
Filters applied
Proportion of features eclairage kept after filtering:
filtered
1    0.845927
0    0.154073
Name: proportion, dtype: float64
Filtered data saved for attribute: eclairage in format: parquet
Filtered data saved for attribute: eclairage in format: csv
Filtered data saved for attribute: eclairage in format: gpkg


**Attribut Confort thermique**

In [16]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----

attribute = 'temperature'
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

with rasterio.open(raster_path) as src:
    # Read the raster data
    raster_data = src.read(1)  # Read first band
    
    # Define NoData
    nodata_val = src.nodata if src.nodata is not None else -9999
    
    # Valid mask: remove NoData
    valid_mask = (raster_data != nodata_val)
    
    # Get coordinates of valid points
    rows, cols = np.where(valid_mask)
    
    # Keep native raster resolution for parquet export
    rows, cols = rows[::10], cols[::10]
    
    # Get coordinates in map units
    xs, ys = rasterio.transform.xy(src.transform, rows, cols)
    temp_values = raster_data[rows, cols]
    
    # Create GeoDataFrame directly with filtered points
    gdf = gpd.GeoDataFrame({
        'temperature': temp_values,
        'filtered': 1,  # All points are filtered since we pre-filtered the data
        'geometry': [Point(x, y) for x, y in zip(xs, ys)]
    }, geometry='geometry')
    
    # Set CRS and convert to target CRS
    gdf = gdf.set_crs(src.crs)
    gdf = gdf.to_crs(target_crs)

print(f"Processing attribute: {attribute}")
print("\nDescriptive statistics:")
print(gdf['temperature'].describe())

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: temperature

Descriptive statistics:
count    406794.000000
mean         31.492273
std          10.037497
min           0.000000
25%          30.278126
50%          33.369716
75%          36.936945
max          54.971017
Name: temperature, dtype: float64
Filtered data saved for attribute: temperature in format: parquet
Filtered data saved for attribute: temperature in format: csv
Filtered data saved for attribute: temperature in format: gpkg


**Lac et cours d'eau**

In [17]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'eau'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)


Data initialized
Processing attribute: eau
Filters applied
Proportion of features eau kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: eau in format: parquet
Filtered data saved for attribute: eau in format: csv
Filtered data saved for attribute: eau in format: gpkg


**Attribut Canopée**

GG : couverture du sol SITG (`SIPV_MN_CARTO_GG.gpkg`, colonne `vegetation`), forêts et végétation arborée uniquement, découpée au périmètre Grand Genève (même logique que 2_1 walk). GE : ancienne couche OSM.

In [18]:
import difflib
import importlib.util
import pyogrio
import shapely


def load_land_cover_classes(gpkg_path, layer, class_column, classes, perimetre_path):
    """
    Lit dans un GeoPackage de couverture du sol (structure SIPV_MN_CARTO_GG) les
    polygones des classes demandées, découpés au périmètre Grand Genève.
    Retourne une GeoDataFrame (colonne `classe`) en operation_crs.
    """
    # Couche à lire
    layers = pyogrio.list_layers(gpkg_path)
    if layer is None:
        if len(layers) != 1:
            raise ValueError(f"Plusieurs couches dans {gpkg_path} : {layers[:, 0].tolist()} — précisez-en une.")
        layer = layers[0][0]

    # Vérification des classes (lecture de la seule colonne, sans géométries)
    classes_df = pyogrio.read_dataframe(
        gpkg_path,
        sql=f'SELECT "{class_column}" AS classe, COUNT(*) AS n FROM "{layer}" GROUP BY "{class_column}" ORDER BY n DESC',
        read_geometry=False,
    )
    available = classes_df["classe"].dropna().astype(str).tolist()
    unknown = [c for c in classes if c not in available]
    for c in unknown:
        print(f"⚠ classe introuvable : {c!r} — proches : {difflib.get_close_matches(c, available, n=3, cutoff=0.5)}")
    if unknown:
        print("Classes disponibles :", available)
        raise ValueError("Corrigez la liste des classes avec des valeurs exactes de la liste ci-dessus.")

    # Périmètre GG (CRS métrique) et son emprise dans le CRS du gpkg (préfiltre spatial)
    perimetre = gpd.read_file(perimetre_path).to_crs(operation_crs)
    perimetre_geom = shapely.make_valid(perimetre.union_all())
    gpkg_crs = pyogrio.read_info(gpkg_path, layer=layer)["crs"]
    bbox = tuple(gpd.GeoSeries([perimetre_geom], crs=operation_crs).to_crs(gpkg_crs).total_bounds)

    # Lecture filtrée dans GDAL : classes choisies + emprise du périmètre
    quoted = ", ".join("'" + c.replace("'", "''") + "'" for c in classes)
    gdf = pyogrio.read_dataframe(
        gpkg_path, layer=layer, columns=[class_column],
        where=f'"{class_column}" IN ({quoted})',
        bbox=bbox,
        use_arrow=importlib.util.find_spec("pyarrow") is not None,
    )
    gdf = gdf.rename(columns={class_column: "classe"})
    print(f"  {len(gdf):,} polygones lus pour {len(classes)} classes")

    # Nettoyage (valides, polygonales, non vides) puis découpe au périmètre
    def _clean(g):
        g["geometry"] = shapely.make_valid(g.geometry.values)
        g = g.explode(index_parts=False, ignore_index=True)
        return g[g.geom_type.isin(["Polygon", "MultiPolygon"]) & ~g.is_empty].copy()

    gdf = _clean(gdf).to_crs(operation_crs)
    n_avant = len(gdf)
    gdf = _clean(gpd.clip(gdf, perimetre_geom))
    print(f"  Découpe au périmètre GG : {n_avant:,} -> {len(gdf):,} polygones")
    print(gdf.groupby("classe").geometry.apply(lambda g: g.area.sum() / 1e4).rename("surface_ha").round(1))
    return gdf


# Réglages communs aux couches de couverture du sol
LAND_COVER_LAYER = None          # None = détection auto s'il n'y a qu'une couche
LAND_COVER_CLASS_COLUMN = "vegetation"
PERIMETRE_GG = f"{input_file_path}/network_agreg/AGGLO_PERIMETRE_AVEC_LAC-SHP"


In [19]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'canopee'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
print(f"Processing attribute: {attribute}")

if territory == 'GG':
    ###----change here----------------------
    # Couverture du sol SITG (chemin et fichier lus dans l'Excel) : végétation arborée uniquement
    CANOPEE_CLASSES = [
        "Forêts fermées de conifères",
        "Forêts fermées de feuillus",
        "Forêts ouvertes",
        "Forêts humides",
        "végétation urbaine",
        "Végétation arborée perturbée",
    ]
    ###--------------------------------------
    gdf = load_land_cover_classes(
        f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}",
        LAND_COVER_LAYER, LAND_COVER_CLASS_COLUMN, CANOPEE_CLASSES, PERIMETRE_GG,
    ).to_crs(target_crs)
else:
    # GE : ancienne couche OSM (landuse=forest, natural=wood)
    gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
    gdf = gdf.to_crs(target_crs)
    gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
    gdf = gdf[gdf.geometry.notna()].copy()

# Filter
gdf['filtered'] = 0
###----change here----------------------
gdf['filtered'] = 1   # la sélection par classe est déjà faite à la lecture
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: canopee


/Users/maximiliantrique/Documents/GitHub/projet_pieton_gg/venv/lib/python3.11/site-packages/pyogrio/core.py:130: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_list_layers(get_vsi_path_or_buffer(path_or_buffer))
/Users/maximiliantrique/Documents/GitHub/projet_pieton_gg/venv/lib/python3.11/site-packages/pyogrio/core.py:279: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read_info(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/contextlib.py:137: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return next(self.gen)


  242,423 polygones lus pour 6 classes
  Découpe au périmètre GG : 242,426 -> 129,164 polygones
classe
Forêts fermées de conifères     13231.6
Forêts fermées de feuillus      64644.5
Forêts humides                   2423.8
Forêts ouvertes                  2751.8
Végétation arborée perturbée     4609.4
végétation urbaine              14112.0
Name: surface_ha, dtype: float64
Filters applied
Proportion of features canopee kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: canopee in format: parquet
Filtered data saved for attribute: canopee in format: csv
Filtered data saved for attribute: canopee in format: gpkg


**Attribut Environnement ouvert naturel**

GG : même couverture du sol que la canopée, classes prairies, cultures, vigne et verger. Pas de routine pour GE.

In [20]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'environnement_ouvert_naturel'
###--------------------

if territory != 'GG':
    print(f"{attribute} : pas de routine pour {territory}")
else:
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    print(f"Processing attribute: {attribute}")

    ###----change here----------------------
    # Même couverture du sol que la canopée : milieux ouverts naturels et agricoles
    ENV_OUVERT_CLASSES = [
        "Prairies humides",
        "Prairies sèches",
        "Prairies extensives",
        "Prairies d'altitudes",
        "Cultures et prairies intensives",
        "Vigne",
        "Verger",
    ]
    ###--------------------------------------
    gdf = load_land_cover_classes(
        f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}",
        LAND_COVER_LAYER, LAND_COVER_CLASS_COLUMN, ENV_OUVERT_CLASSES, PERIMETRE_GG,
    ).to_crs(target_crs)

    # Filter
    gdf['filtered'] = 0
    ###----change here----------------------
    gdf['filtered'] = 1   # la sélection par classe est déjà faite à la lecture
    ###--------------------------------------
    print("Filters applied")

    # Count and proportion
    print(f"Proportion of features {attribute} kept after filtering:")
    print(gdf.filtered.value_counts(normalize=True))

    # Save
    save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: environnement_ouvert_naturel


/Users/maximiliantrique/Documents/GitHub/projet_pieton_gg/venv/lib/python3.11/site-packages/pyogrio/core.py:130: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_list_layers(get_vsi_path_or_buffer(path_or_buffer))
/Users/maximiliantrique/Documents/GitHub/projet_pieton_gg/venv/lib/python3.11/site-packages/pyogrio/core.py:279: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read_info(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/contextlib.py:137: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return next(self.gen)


  89,683 polygones lus pour 7 classes
  Découpe au périmètre GG : 89,683 -> 37,908 polygones
classe
Cultures et prairies intensives    37850.7
Prairies d'altitudes                9495.0
Prairies extensives                19469.0
Prairies humides                     415.8
Prairies sèches                     1198.9
Verger                               563.9
Vigne                               2680.7
Name: surface_ha, dtype: float64
Filters applied
Proportion of features environnement_ouvert_naturel kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: environnement_ouvert_naturel in format: parquet
Filtered data saved for attribute: environnement_ouvert_naturel in format: csv
Filtered data saved for attribute: environnement_ouvert_naturel in format: gpkg


**Attribut pollution air**

In [21]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'air'

###--------------------

    
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

with rasterio.open(raster_path) as src:
    # Read the raster data
    raster_data = src.read(1)  # Read first band
    
    # Define NoData
    nodata_val = src.nodata if src.nodata is not None else -9999
    
    # Valid mask: remove NoData
    valid_mask = (raster_data != nodata_val)
    
    # Get coordinates of valid points
    rows, cols = np.where(valid_mask)
    
    # Keep air sampling denser than before, but lighter than temperature
    rows, cols = rows[::1], cols[::1]
    
    # Get coordinates in map units
    xs, ys = rasterio.transform.xy(src.transform, rows, cols)
    temp_values = raster_data[rows, cols]
    
    # ✅ Conversion en µmol/m² pour éviter les underflows d'affichage
    temp_values_converted = temp_values * 1e6  # mol/m² → µmol/m²
    
    gdf = gpd.GeoDataFrame({
        'no_2': temp_values_converted,   # valeurs lisibles ex: 18 à 40 µmol/m²
        'no_2_raw': temp_values,         # valeurs originales conservées si besoin
        'filtered': 1,
        'geometry': [Point(x, y) for x, y in zip(xs, ys)]
    }, geometry='geometry')
    
    gdf = gdf.set_crs(src.crs)
    gdf = gdf.to_crs(target_crs)

print(f"Processing attribute: {attribute}")
print("\nDescriptive statistics:")
print(gdf['no_2'].describe())

# Save
print(f"Saving {attribute}: can take up to 3mn")
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: air

Descriptive statistics:
count    5355.000000
mean       30.010011
std         3.194511
min        17.885202
25%        27.770937
50%        29.925370
75%        31.997349
max        40.489754
Name: no_2, dtype: float64
Saving air: can take up to 3mn
Filtered data saved for attribute: air in format: parquet
Filtered data saved for attribute: air in format: csv
Filtered data saved for attribute: air in format: gpkg


**Attribut Infrastructure cyclable** (remplace `piste` et `bande`)

Lit la classe `infra_cyclable` des arêtes de `bike_edges_graph` (0_1 : tags OSM ∪ schéma cyclable d'agglomération) et la convertit en score.

| `infra_cyclable` | score |
| --- | --- |
| piste_cyclable | 1 |
| bande_cyclable | 0.7 |
| voie_mixte | 0.7 |
| footway_designe | 0.6 |
| partage_bus | 0.5 |
| accotement | 0.5 |
| bike_pedestrian_area | 0.4 |
| partage | 0.3 |
| bike_road | 0 |

Classes ajoutées le 2026-11-19 (définies dans `map_bike_infra`, 0_1) :
- `voie_mixte` : voie vélo partagée avec les piétons (`highway=path` + `bicycle=designated` non séparé, `highway=cycleway` + `segregated=no`) ;
- `footway_designe` : `highway=footway` + `bicycle=designated` ;
- `accotement` : `cycleway=shoulder`.
Les `highway=footway` + `bicycle=yes/permissive` passent en `bike_pedestrian_area`.

Agrégation en 2_2 : `how = rectangle_weighted_mean` sur `infra_cyclable_score`, demi-largeur 2 m : le segment prend le score de sa propre arête (les deux sens d'une arête ont la même classe).

In [6]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'infra_cyclable'
INFRA_SCORES = {
    "piste_cyclable": 1.0,
    "bande_cyclable": 0.7,
    "voie_mixte": 0.7,         # >>> ajouté 2026-11-19
    "footway_designe": 0.6,    # >>> ajouté 2026-11-19
    "partage_bus": 0.5,
    "accotement": 0.5,         # >>> ajouté 2026-11-19
    "bike_pedestrian_area": 0.4,
    "partage": 0.3,
    "bike_road": 0.0,
}
VALUE_COLUMN = "infra_cyclable_score"   # -> value_column_GG de l'Excel
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

if "infra_cyclable" not in gdf.columns:
    raise KeyError("infra_cyclable absent de bike_edges_graph : relancer 0_1 (section 7, schéma cyclable)")

unknown = sorted(set(gdf["infra_cyclable"].dropna()) - set(INFRA_SCORES))
if unknown:
    print(f"⚠️ Classes sans score (mises à 0) : {unknown}")
gdf[VALUE_COLUMN] = gdf["infra_cyclable"].map(INFRA_SCORES).fillna(0.0)

# Filter
gdf['filtered'] = 0
###----change here----------------------
gdf['filtered'] = 1   # toutes les arêtes : bike_road compte à 0 dans la moyenne
###--------------------------------------
print("Filters applied")

# Contrôle : longueur par classe
L = gdf.to_crs(operation_crs).length / 1000
print((L.groupby(gdf["infra_cyclable"]).sum()).round(1).rename("km").to_string())
if "infra_cyclable_source" in gdf.columns:
    print(gdf["infra_cyclable_source"].value_counts().to_string())

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: infra_cyclable
Filters applied
infra_cyclable
accotement                 72.5
bande_cyclable            624.8
bike_pedestrian_area      579.4
bike_road               10317.1
footway_designe            92.9
partage                 12597.5
partage_bus                74.7
piste_cyclable            883.3
voie_mixte                141.4
infra_cyclable_source
osm       198048
schema      5825
Filtered data saved for attribute: infra_cyclable in format: parquet
Filtered data saved for attribute: infra_cyclable in format: csv
Filtered data saved for attribute: infra_cyclable in format: gpkg


**Attribut environnement agréable (alentours)**

In [23]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'alentours'


###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
landuse_agreable = {
    # 3 = très agréable
    "forest",
    "grass",
    "meadow",
    "recreation_ground"
}

gdf.loc[gdf["landuse"].isin(landuse_agreable), "filtered"] = 1


###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: alentours
Filters applied
Proportion of features alentours kept after filtering:
filtered
1    0.84608
0    0.15392
Name: proportion, dtype: float64
Filtered data saved for attribute: alentours in format: parquet
Filtered data saved for attribute: alentours in format: csv
Filtered data saved for attribute: alentours in format: gpkg


**Attribut conflits usage**

In [24]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'conflit_md'

row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")


# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
gdf['severity_score'] = 0
gdf['severity_score'] = gdf['severity'].map({'high' : 3, 'medium':2, 'low':1})

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: conflit_md
Filters applied
Proportion of features conflit_md kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: conflit_md in format: parquet
Filtered data saved for attribute: conflit_md in format: csv
Filtered data saved for attribute: conflit_md in format: gpkg


**Attribut service vélo**

In [25]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'service_velo'

row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")


# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: service_velo
Filters applied
Proportion of features service_velo kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: service_velo in format: parquet
Filtered data saved for attribute: service_velo in format: csv
Filtered data saved for attribute: service_velo in format: gpkg


**Attribut parking abris**

In [26]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'parking_abris'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf["covered"] == "yes", "filtered"] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: parking_abris
Filters applied
Proportion of features parking_abris kept after filtering:
filtered
0    0.792929
1    0.207071
Name: proportion, dtype: float64
Filtered data saved for attribute: parking_abris in format: parquet
Filtered data saved for attribute: parking_abris in format: csv
Filtered data saved for attribute: parking_abris in format: gpkg


**Attribut accident**

In [27]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----

attribute = 'accident'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
print(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
#gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gpd.read_file(
    f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}",
    encoding="latin1"   # or "ISO-8859-1"
)
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------

gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
../../Data/input/attributs/GG/Merged_Accidents_2019_2023/Merged_GG_2019_2023.shp
Processing attribute: accident
Filters applied
Proportion of features accident kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: accident in format: parquet
Filtered data saved for attribute: accident in format: csv
Filtered data saved for attribute: accident in format: gpkg


**Attribut charge de trafic (TIM OFROU)**

Couche linéaire du modèle de trafic (plan de charges TI, jour moyen, état actuel), une ligne par lien et par sens. Valeur : `VOLVEHPRT(AP)` (tronquée en `VOLVEHPR~1` dans le .dbf), renommée `volvehprt_ap`.

Avec `TWO_WAY = True`, les deux sens d'un même lien (`NO`, géométries inversées) sont additionnés : on obtient la charge des deux sens sur une seule ligne. Sinon, la moyenne en 2_2 donnerait la charge d'un seul sens.

Agrégation en 2_2 : `how_GG = rectangle_weighted_mean` (`feature_rectangle.py`). Rectangle à extrémités plates autour du segment, demi-largeur `buffer_size_GG`, moyenne des lignes pondérée par leur longueur dans le rectangle ; les lignes obliques (> 45°) sont écartées.

In [16]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'charge'
TWO_WAY = True                     # additionner les deux sens d'un même lien (NO)
VALUE_COLUMN = "VOLVEHPR~1"      # nom de sortie -> value_column_GG de l'Excel
###--------------------

row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
if pd.isna(row[attribute_source_path]) or row[attribute_source_path] == 'unavailable':
    print(f"{attribute} non disponible pour le territoire {territory}")
else:
    path = f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}"
    gdf = gpd.read_file(path)   # CRS du .prj : Sphere_Mercator -> reprojeté plus bas
    print(f"Processing attribute: {attribute} ({len(gdf):,} liens orientés)")

    # Colonne de charge : VOLVEHPRT(AP), tronquée par le format .dbf (VOLVEHPR~1)
    vol_cols = [c for c in gdf.columns if c.upper().startswith("VOLVEHPR")]
    if len(vol_cols) != 1:
        raise ValueError(f"Colonne VOLVEHPRT(AP) introuvable ou ambiguë : {list(gdf.columns)}")
    gdf = gdf.rename(columns={vol_cols[0]: VALUE_COLUMN})
    gdf[VALUE_COLUMN] = pd.to_numeric(gdf[VALUE_COLUMN], errors="coerce")
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].to_crs(operation_crs)

    if TWO_WAY:
        # Un lien = deux lignes (une par sens, géométrie inversée) : on somme les sens
        n_sens = gdf.groupby("NO").size()
        print(f"  Liens à deux sens : {int((n_sens == 2).sum()):,} | à un sens : {int((n_sens == 1).sum()):,}")
        gdf = gdf.groupby("NO", as_index=False).agg(
            **{VALUE_COLUMN: (VALUE_COLUMN, "sum"),
               "n_sens": (VALUE_COLUMN, "size"),
               "TYPENO": ("TYPENO", "first"),
               "geometry": ("geometry", "first")})
        gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=operation_crs)
    gdf = gdf.to_crs(target_crs)

    # Filter
    gdf['filtered'] = 0
    ###----change here----------------------
    gdf.loc[gdf[VALUE_COLUMN].notna(), 'filtered'] = 1
    ###--------------------------------------
    print("Filters applied")

    print(f"Proportion of features {attribute} kept after filtering:")
    print(gdf.filtered.value_counts(normalize=True))
    print(f"\n{VALUE_COLUMN} (véh./jour{', deux sens' if TWO_WAY else ', par sens'}) :")
    print(gdf[VALUE_COLUMN].describe().round(0))

    # Save
    save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: charge (26,685 liens orientés)
  Liens à deux sens : 12,568 | à un sens : 1,549
Filters applied
Proportion of features charge kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64

VOLVEHPR~1 (véh./jour, deux sens) :
count    14117.0
mean      5974.0
std       7697.0
min          0.0
25%        884.0
50%       3459.0
75%       7766.0
max      56413.0
Name: VOLVEHPR~1, dtype: float64
Filtered data saved for attribute: charge in format: parquet
Filtered data saved for attribute: charge in format: csv
Filtered data saved for attribute: charge in format: gpkg


**Attribut Traversées (vélo)**

Même méthode que la marche (2_1 walk), avec une seule différence : pour le vélo, seules les traversées **`crossing=traffic_signals`** sont retenues (`BIKE_CROSSING_VALUES`, passé à `filter_crossings`).

Points OSM `walk_crossing_nodes.parquet` (0_1) sauvés **deux fois**, avec un `filtered` différent :
- `n_traversees` : `filtered = 1` si la traversée est retenue (ici : à feux) ;
- `n_traversees_feux` : `filtered = 1` si elle est retenue **et** à feux (`SIGNAL_RULES`). Avec `ADD_TRAFFIC_SIGNAL_NODES = True`, on ajoute les nœuds `highway=traffic_signals` (feux de carrefour, couche `traffic_signals` de `osm_attributes.gpkg`), en écartant ceux à moins de `SIGNAL_MERGE_DIST_M` d'un feu déjà retenu. Colonne `source_noeud` pour les distinguer dans QGIS.

Avec le filtre vélo, les deux couches sont identiques ; on garde les deux pour réutiliser tel quel le score de `derived_attributes.py`.

Comptage en 2_2 (`how = count_rectangle`), score `traversees` dans la section « Attributs dérivés » de 2_2, avec la vitesse vélo de 1_1 (`vitesse_bike_edges.parquet`). Implémenté pour GG uniquement.

In [10]:
# ============================================================================
# Traversées vélo : points OSM -> 2 couches (n_traversees, n_traversees_feux)
#   Mêmes règles et même code que la marche (crossing_filters.py), mais seules
#   les traversées crossing=traffic_signals sont retenues.
# ============================================================================
import importlib
import crossing_filters
importlib.reload(crossing_filters)
from crossing_filters import load_crossings, filter_crossings

# Initialize
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute_all = "n_traversees"        # traversées retenues
attribute_sig = "n_traversees_feux"   # traversées retenues à feux
BIKE_CROSSING_VALUES = ["traffic_signals"]   # vélo : feux uniquement (marche : CROSSING_VALUES_KEPT)

# Feux de carrefour : nœuds highway=traffic_signals (couche « traffic_signals » de 0_2),
# souvent posés sur le carrefour ou la ligne d'arrêt, sans nœud crossing à côté
ADD_TRAFFIC_SIGNAL_NODES = True
SIGNALS_GPKG = f"{input_file_path}/attributs/{territory}/osm/osm_attributes.gpkg"
SIGNALS_LAYER = "traffic_signals"
SIGNAL_MERGE_DIST_M = 5.0   # fusionne les feux à moins de X m d'un feu déjà retenu (0 = pas de fusion)
###--------------------

if territory == 'GE':
    print("traversees : pas de routine pour GE")

if territory == 'GG':
    row_all = attributs_info[attributs_info['attribute'] == attribute_all].iloc[0]
    row_sig = attributs_info[attributs_info['attribute'] == attribute_sig].iloc[0]
    # Check availability
    if row_all[attribute_source_path] == 'unavailable':
        print(f"{attribute_all} non disponible pour le territoire {territory}")
    else:
        crossings_path = f"{input_file_path}/{row_all[attribute_source_path]}/{row_all[file_name]}"
        print(f"Processing attributes: {attribute_all}, {attribute_sig}  ({crossings_path})")
        gdf = filter_crossings(load_crossings(crossings_path, target_crs),
                               values_kept=BIKE_CROSSING_VALUES, drop=False)
        gdf = gdf[gdf.geometry.notna()].copy()
        gdf["source_noeud"] = "highway=crossing"

        if ADD_TRAFFIC_SIGNAL_NODES:
            sig = gpd.read_file(SIGNALS_GPKG, layer=SIGNALS_LAYER).to_crs(target_crs)
            sig = sig[sig.geometry.notna() & sig.geom_type.eq("Point")]
            sig = gpd.GeoDataFrame({
                "osmid": sig["osmid"] if "osmid" in sig.columns else range(len(sig)),
                "crossing_tag": "traffic_signals",
                "crossing_kept": True,
                "is_signal": True,
                "source_noeud": "highway=traffic_signals",
            }, geometry=sig.geometry.values, crs=target_crs)

            # Un nœud déjà présent comme traversée n'est pas compté deux fois
            if "osmid" in gdf.columns:
                sig = sig[~sig["osmid"].astype(str).isin(gdf["osmid"].astype(str))]

            # Fusion spatiale : feux à moins de SIGNAL_MERGE_DIST_M d'une traversée à feux
            # ou d'un autre feu déjà retenu (plusieurs nœuds pour un même carrefour)
            if SIGNAL_MERGE_DIST_M > 0 and len(sig):
                ref = gdf.loc[gdf["crossing_kept"] & gdf["is_signal"]].to_crs(operation_crs)
                sig_m = sig.to_crs(operation_crs)
                near = gpd.sjoin_nearest(sig_m, ref[["geometry"]], how="left",
                                         max_distance=SIGNAL_MERGE_DIST_M).index_right.notna()
                near = near.groupby(level=0).any()
                sig_m = sig_m[~near.reindex(sig_m.index, fill_value=False)]
                keep, kept_pts = [], []
                for i, pt in zip(sig_m.index, sig_m.geometry):   # dédoublonnage entre feux
                    if all(pt.distance(q) > SIGNAL_MERGE_DIST_M for q in kept_pts):
                        keep.append(i); kept_pts.append(pt)
                sig = sig.loc[keep]

            print(f"Feux de carrefour ajoutés : {len(sig):,} (couche {SIGNALS_LAYER})")
            gdf = gpd.GeoDataFrame(pd.concat([gdf, sig], ignore_index=True), geometry="geometry", crs=target_crs)

        print(gdf.loc[gdf["crossing_kept"]].groupby("source_noeud").size().rename("noeuds retenus").to_string())

        # Filter
        ###----change here----------------------
        gdf_all = gdf.copy()
        gdf_all['filtered'] = gdf_all['crossing_kept'].astype(int)

        gdf_sig = gdf.copy()
        gdf_sig['filtered'] = (gdf_sig['crossing_kept'] & gdf_sig['is_signal']).astype(int)
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        for name, g in [(attribute_all, gdf_all), (attribute_sig, gdf_sig)]:
            print(f"\nProportion of features {name} kept after filtering:")
            print(g.filtered.value_counts(normalize=True))

        # Save (deux couches, une par ligne de l'Excel)
        save(save_filtered_attributes, row_all, gdf_all, attribute_all)
        save(save_filtered_attributes, row_sig, gdf_sig, attribute_sig)

Data initialized
Processing attributes: n_traversees, n_traversees_feux  (../../Data/input/networkGG/walk_crossing_nodes.parquet)
Valeurs du tag crossing (avant filtre) :
crossing
uncontrolled                    8168
None                            2621
marked                          2280
unmarked                        2079
traffic_signals                 1820
zebra                            185
informal                          11
pavement                           2
uncontrolled;traffic_signals       1
island                             1

Traversées retenues : 1,820 / 17,168 (dont à feux : 1,820)
Feux de carrefour ajoutés : 1,560 (couche traffic_signals)
source_noeud
highway=crossing           1820
highway=traffic_signals    1560
Filters applied

Proportion of features n_traversees kept after filtering:
filtered
0    0.819522
1    0.180478
Name: proportion, dtype: float64

Proportion of features n_traversees_feux kept after filtering:
filtered
0    0.819522
1    0.180478
Name: pr

**Attribut Pente** !!! Depreciated 28.05.26 - cet attribut est maintenant calculé dans 1_1, directement sur le réseau, comme la cennectivité

In [11]:
# # Initialize
# attribute = None
# row = None
# gdf = None
# print("Data initialized")

# # Load
# ###----change here-----
# attribute = 'pente'

# ###--------------------

    
# row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
# raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

# with rasterio.open(raster_path) as src:
#     # Read the raster data
#     raster_data = src.read(1)  # Read first band
    
#     # Define NoData
#     nodata_val = src.nodata if src.nodata is not None else -9999
    
#     # Valid mask: remove NoData
#     valid_mask = (raster_data != nodata_val)
    
#     # Get coordinates of valid points
#     rows, cols = np.where(valid_mask)
    
#     # Sample every 10th point
#     rows, cols = rows[::10], cols[::10]
    
#     # Get coordinates in map units
#     xs, ys = rasterio.transform.xy(src.transform, rows, cols)
#     temp_values = raster_data[rows, cols]
    
#     # Create GeoDataFrame directly with filtered points
#     gdf = gpd.GeoDataFrame({
#         'slope_degree': temp_values,
#         'filtered': 1,  # All points are filtered since we pre-filtered the data
#         'geometry': [Point(x, y) for x, y in zip(xs, ys)]
#     }, geometry='geometry')
    
#     # Set CRS and convert to target CRS
#     gdf = gdf.set_crs(src.crs)
#     gdf = gdf.to_crs(target_crs)

# print(f"Processing attribute: {attribute}")
# print("\nDescriptive statistics:")
# print(gdf['slope_degree'].describe())

# # Save
# print(f"Saving {attribute}: can take up to 3mn")
# save(save_filtered_attributes, row, gdf, attribute)

**Attribut tourner à droite**

In [12]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'tourner_droite'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------

gdf.loc[gdf['TYPE_FEUX'] == 'plaq-comp-droite', 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: tourner_droite
Filters applied
Proportion of features tourner_droite kept after filtering:
filtered
0    0.991557
1    0.008443
Name: proportion, dtype: float64
Filtered data saved for attribute: tourner_droite in format: parquet
Filtered data saved for attribute: tourner_droite in format: csv
Filtered data saved for attribute: tourner_droite in format: gpkg
